# BIO Annotation Pipeline — per-species output

Converts every species JSON file in `all_species_json_final_dataset/` (which carries
`[CODE = value]` audit brackets inside `corrected_text_final`) into a
per-species BIO-tagged JSONL file written to `../bio_annotation_final_dataset/`.

Output file: `<id>_<specie>_bio.json`, one compact JSON line per sentence:
`{"tokens": [...], "biotags": [...]}`

**Entity whitelist:** `T, Wwi, am, Wwb, Li, Lp, F_R_i, N_R_i, tp, ab, tx, Wwp, Ri, tR, Lb, tg, JOi, Ww0, GSI, Wwx`

**Note on the offset fix:** an earlier version of this pipeline (`ner_pipeline.ipynb`)
removed `[CODE = value]` brackets and then ran a second cosmetic pass (collapsing
stray double-spaces and removing spaces-before-punctuation left over in the
source text) *without* re-tracking the character shift from that second pass.
Any entity occurring after such a cosmetic edit landed on the wrong token(s)
(e.g. `7% ` -> `%` tagged instead of `7`). This notebook fixes that by tracking
every deleted character — brackets AND cosmetic edits — through a single
keep-mask, so span offsets stay correct end-to-end.

## Cell 0 – Imports and configuration

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import spacy
from tqdm import tqdm

# Notebook lives at: data_annotation/data_annotation_notebook/bio_annotation_all_species.ipynb
DATA_DIR = Path("../all_species_json_final_dataset") 
OUT_DIR  = Path("../bio_annotation_final_dataset")
OUT_DIR.mkdir(exist_ok=True)

WHITELIST = {
    "T", "Wwi", "am", "Wwb", "Li", "Lp",
    "F_R_i", "N_R_i", "tp", "ab", "tx",
    "Wwp", "Ri", "tR", "Lb", "tg", "JOi",
    "Ww0", "GSI", "Wwx"
}

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "lemmatizer", "tagger", "attribute_ruler"])
nlp.add_pipe("sentencizer")

print("Setup complete. spaCy", spacy.__version__)


Setup complete. spaCy 3.8.7


## Cell 1 – Load all species records

In [2]:
records = []
for fpath in sorted(DATA_DIR.glob("*.json")):
    d = json.loads(fpath.read_text(encoding="utf-8"))
    records.append(d)

print(f"Loaded {len(records):,} species files")
print("Example:", records[0]["id"], records[0]["specie"], records[0]["specie_en"])


Loaded 7,281 species files
Example: 1 Abatus_cordatus Kerguelen sea urchin


## Cell 2 – Bracket parsing & preceding-mention span extraction

In [3]:
# Locate the natural-language mention preceding each [CODE = value] bracket.
# Strategy: verbatim search of the bracket's value string in the 400-char
# window before '[' (rightmost match); fall back to the numeric token if the
# value isn't restated verbatim.

BRACKET_RE = re.compile(r'\[([^\]\[=]+?)\s*=\s*([^\]\[]+?)\]')
_INNER_BRACKET_RE = re.compile(r'\[[^\]]*\]')


def _clean_bracket_value(raw_val):
    return _INNER_BRACKET_RE.sub('', raw_val).strip()


def _extract_numeric_tokens(s):
    return re.findall(r'\d+(?:[.,]\d+)*', s)


def find_preceding_span(text, bracket_start, value, window=400):
    clean_val = _clean_bracket_value(value)
    win_start = max(0, bracket_start - window)
    window_text = text[win_start:bracket_start]

    pos = window_text.rfind(clean_val)
    if pos != -1:
        abs_start = win_start + pos
        return abs_start, abs_start + len(clean_val)

    nums = _extract_numeric_tokens(clean_val)
    if nums:
        first_num = re.escape(nums[0])
        pattern = re.compile(rf'{first_num}\s*(?:[°a-zA-Z%/]+(?:\s*[a-zA-Z°%/]+)*)?')
        matches = list(pattern.finditer(window_text))
        if matches:
            m = matches[-1]
            abs_start = win_start + m.start()
            return abs_start, win_start + m.end()
    return None


def extract_raw_spans(text):
    spans = []
    for m in BRACKET_RE.finditer(text):
        code_ = m.group(1).strip()
        val = m.group(2)
        if code_ not in WHITELIST:
            continue
        span = find_preceding_span(text, m.start(), val)
        if span is None:
            continue
        spans.append((span[0], span[1], code_))
    return spans


## Cell 3 – Clean text & rebase span offsets (bug-fixed)

In [4]:
# Remove [CODE = value] brackets, then collapse stray double-spaces and
# spaces-before-punctuation left in the source text. Every deletion (bracket
# OR cosmetic) is tracked in a single per-character keep-mask, so entity
# offsets rebase correctly no matter how many cosmetic edits precede them.

ALL_BRACKET_RE = re.compile(r'\s*\[[^\[\]]+\]')


def clean_text_and_rebase(raw_text, raw_spans):
    chars = list(raw_text)
    pos_map = list(range(len(raw_text)))   # pos_map[i] = raw index of chars[i]

    # Pass 1: strip [CODE = value] brackets (with leading whitespace)
    keep = [True] * len(chars)
    for m in ALL_BRACKET_RE.finditer(raw_text):
        for i in range(m.start(), m.end()):
            keep[i] = False
    chars = [c for c, k in zip(chars, keep) if k]
    pos_map = [p for p, k in zip(pos_map, keep) if k]

    # Pass 2: collapse double-spaces -> single space (keep first, drop rest)
    text1 = ''.join(chars)
    keep2 = [True] * len(chars)
    for m in re.finditer(r'  +', text1):
        for i in range(m.start() + 1, m.end()):
            keep2[i] = False
    chars = [c for c, k in zip(chars, keep2) if k]
    pos_map = [p for p, k in zip(pos_map, keep2) if k]

    # Pass 3: remove space before punctuation
    text2 = ''.join(chars)
    keep3 = [True] * len(chars)
    for m in re.finditer(r' ([,\.;:!?])', text2):
        keep3[m.start()] = False
    chars = [c for c, k in zip(chars, keep3) if k]
    pos_map = [p for p, k in zip(pos_map, keep3) if k]

    joined = ''.join(chars)
    lead_strip = len(joined) - len(joined.lstrip())
    trail_strip = len(joined) - len(joined.rstrip())
    if lead_strip:
        pos_map = pos_map[lead_strip:]
    if trail_strip:
        pos_map = pos_map[:len(pos_map) - trail_strip]
    cleaned = joined.strip()

    raw_to_clean = {raw_p: clean_p for clean_p, raw_p in enumerate(pos_map)}

    rebased = []
    for s, e, etype in raw_spans:
        if s not in raw_to_clean or (e - 1) not in raw_to_clean:
            continue   # span touched a deleted character - skip
        new_s = raw_to_clean[s]
        new_e = raw_to_clean[e - 1] + 1
        rebased.append((new_s, new_e, etype))
    return cleaned, rebased


# Apply to every record
clean_texts = []
clean_spans_list = []
for d in records:
    raw_text = d["corrected_text_final"]
    raw_spans = extract_raw_spans(raw_text)
    clean_text, clean_spans = clean_text_and_rebase(raw_text, raw_spans)
    clean_texts.append(clean_text)
    clean_spans_list.append(clean_spans)

# Sanity check on the first species
print("Species:", records[0]["specie"])
for s, e, et in clean_spans_list[0][:6]:
    print(f"  [{et}] {clean_texts[0][s:e]!r}")


Species: Abatus_cordatus
  [Li] '8 cm'
  [GSI] '7%'
  [T] '5 °C'
  [Ww0] '0.001777 grams'
  [Lp] '1.9 cm'
  [Wwp] '2.9 grams'


## Cell 4 – Span-to-token alignment & BIO tagging

In [5]:
def align_span_to_tokens(span_start, span_end, tok_starts, tok_ends):
    try:
        ti = tok_starts.index(span_start)
    except ValueError:
        ti = next((i for i, ts in enumerate(tok_starts) if ts >= span_start and ts < span_end), None)
        if ti is None:
            return None
    try:
        tj = next(i for i in range(len(tok_ends) - 1, -1, -1) if tok_ends[i] == span_end)
    except StopIteration:
        tj = next((i for i in range(len(tok_ends) - 1, -1, -1)
                    if tok_ends[i] <= span_end and tok_starts[i] >= span_start), None)
        if tj is None:
            return None
    if ti > tj:
        return None
    return ti, tj


def resolve_overlaps(aligned_spans):
    # Longer span wins on overlap; ties keep the earlier-starting span.
    spans = sorted(aligned_spans, key=lambda x: (-(x[1] - x[0]), x[0]))
    occupied = set()
    kept = []
    for ti, tj, etype in spans:
        tok_set = set(range(ti, tj + 1))
        if tok_set & occupied:
            continue
        occupied |= tok_set
        kept.append((ti, tj, etype))
    return sorted(kept, key=lambda x: x[0])


## Cell 5 – Tokenise, tag, and write one BIO JSON per species

In [6]:
n_written = 0
n_entities_total = 0
bio_seq_errors = 0
align_failures = 0

docs = nlp.pipe(clean_texts, batch_size=50)

for d, doc, clean_spans in tqdm(zip(records, docs, clean_spans_list), total=len(records),
                                 desc="Tagging species", unit="species"):
    out_lines = []
    for sent in doc.sents:
        toks, starts, ends = [], [], []
        for tok in sent:
            if tok.is_space:
                continue
            toks.append(tok.text)
            starts.append(tok.idx)
            ends.append(tok.idx + len(tok.text))
        if not toks:
            continue

        sent_start, sent_end = starts[0], ends[-1]
        relevant = [(s, e, et) for s, e, et in clean_spans if not (s >= sent_end or e <= sent_start)]

        aligned = []
        for s, e, et in relevant:
            res = align_span_to_tokens(s, e, starts, ends)
            if res is None:
                align_failures += 1
            else:
                aligned.append((res[0], res[1], et))

        tags = ["O"] * len(toks)
        for ti, tj, et in resolve_overlaps(aligned):
            tags[ti] = f"B-{et}"
            for k in range(ti + 1, tj + 1):
                tags[k] = f"I-{et}"

        n_entities_total += sum(1 for t in tags if t.startswith("B-"))

        prev = "O"
        for t in tags:
            if t.startswith("I-"):
                etype = t[2:]
                if prev not in (f"B-{etype}", f"I-{etype}"):
                    bio_seq_errors += 1
            prev = t

        out_lines.append(json.dumps({"tokens": toks, "biotags": tags}, ensure_ascii=False))

    out_path = OUT_DIR / f"{d['id']:04d}_{d['specie']}_bio.json"
    out_path.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
    n_written += 1

print(f"\nSpecies processed   : {n_written:,}")
print(f"Entities tagged     : {n_entities_total:,}")
print(f"Alignment failures  : {align_failures:,}")
print(f"BIO sequence errors : {bio_seq_errors:,}")
print(f"Files written to    : {OUT_DIR.resolve()}")


Tagging species: 100%|██████████| 7281/7281 [04:16<00:00, 28.36species/s]


Species processed   : 7,281
Entities tagged     : 119,527
Alignment failures  : 0
BIO sequence errors : 0
Files written to    : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/bio_annotation_final_dataset


## Cell 6 – Validation: spot-check output files

In [7]:
written_files = sorted(OUT_DIR.glob("*_bio.json"))
print(f"Files in {OUT_DIR}: {len(written_files):,}")

# Show first 2 lines of the first species as a sanity check
sample = written_files[0].read_text(encoding="utf-8").splitlines()
print(f"\n{written_files[0].name} — {len(sample)} sentences")
for line in sample[:2]:
    print(line)


Files in ../bio_annotation_final_dataset: 7,281

0001_Abatus_cordatus_bio.json — 27 sentences
{"tokens": ["Field", "surveys", "at", "Anse", "du", "Halage", "have", "provided", "extensive", "data", "on", "the", "life", "history", "of", "the", "Kerguelen", "sea", "urchin", "Abatus", "cordatus", "."], "biotags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]}
{"tokens": ["Among", "the", "most", "notable", "findings", "is", "the", "maximum", "observed", "length", "recorded", "for", "individuals", "at", "this", "station", ",", "which", "reaches", "4.2", "cm", "."], "biotags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]}


In [8]:
## Cell 7 – Investigate the 1 alignment failure

align_failures_detail = []

docs2 = nlp.pipe(clean_texts, batch_size=50)

for d, doc, clean_spans in zip(records, docs2, clean_spans_list):
    for sent in doc.sents:
        toks, starts, ends = [], [], []
        for tok in sent:
            if tok.is_space:
                continue
            toks.append(tok.text)
            starts.append(tok.idx)
            ends.append(tok.idx + len(tok.text))
        if not toks:
            continue

        sent_start, sent_end = starts[0], ends[-1]
        relevant = [(s, e, et) for s, e, et in clean_spans if not (s >= sent_end or e <= sent_start)]

        for s, e, et in relevant:
            res = align_span_to_tokens(s, e, starts, ends)
            if res is None:
                snippet = clean_texts[records.index(d)][max(0, s-30):e+30]
                align_failures_detail.append({
                    "species": d["specie"],
                    "entity_type": et,
                    "span_text": clean_texts[records.index(d)][s:e],
                    "context": snippet,
                    "span": (s, e),
                })

print(f"Alignment failures  : {len(align_failures_detail)}")
for f in align_failures_detail:
    print(f"\nSpecies      : {f['species']}")
    print(f"Entity type  : {f['entity_type']}")
    print(f"Span text    : {f['span_text']!r}")
    print(f"Span offsets : {f['span']}")
    print(f"Context      : {f['context']!r}")


Alignment failures  : 0


## Cell 7 – Combine all per-species files into one JSONL dataset

Concatenates every `<id>_<specie>_bio.json` file in `OUT_DIR` into a single
`.jsonl` file (one sentence per line, same `{"tokens": [...], "biotags": [...]}`
shape) written to a new `../bio_dataset_combined/` folder.

In [9]:
# 
import json
from pathlib import Path

SOURCE_DIR = Path("../bio_annotation_final_dataset")
COMBINED_DIR = Path("../animalia_dataset")
COMBINED_DIR.mkdir(exist_ok=True)
combined_path = COMBINED_DIR / "all_species_bio.jsonl"

bio_files = sorted(SOURCE_DIR.glob("*_bio.json"))
n_species = 0
n_sentences = 0

with combined_path.open("w", encoding="utf-8") as out_f:
    for fpath in bio_files:
        for line in fpath.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                rec["source"] = fpath.name
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                n_sentences += 1
        n_species += 1

print(f"Species combined   : {n_species:,}")
print(f"Sentences written  : {n_sentences:,}")
print(f"Combined file      : {combined_path.resolve()}")

Species combined   : 7,281
Sentences written  : 191,337
Combined file      : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/all_species_bio.jsonl
